# 🔬 RedPitayaSTCL: Setup, Compatibility Check & Auto-Install

<p style="font-size:1.05em; color:#444;">
This notebook verifies that your <strong>PC environment</strong> and each
<strong>RedPitaya board</strong> meet all requirements before running the
Scanning Transfer Cavity Lock (STCL) system.
<br><br>
<strong style="color:#27ae60;">New:</strong> Missing PC packages are installed automatically.
Missing board files are uploaded and fixable board issues are resolved automatically.
</p>

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong>→ Run all cells top-to-bottom on a fresh kernel before any locking workflow. Remember to update ip-address in the code cell below.</strong>
</blockquote>

<br>

| Step | What it does |
|------|-------------|
| **①** | Configuration — set board IPs here |
| **②** | PC auto-install — installs any missing packages, then imports checker |
| **③** | PC environment checks |
| **④** | Board information (SSH) |
| **⑤** | Board compatibility checks + auto-fix |
| **⑥** | Summary report |

---
## 1) Configuration

Edit the cell below **before running anything else**. Add one entry per physical RedPitaya board.

| Key | Example | Description |
|-----|---------|-------------|
| `ip` | `"192.168.0.101"` | Board IP address on your network |
| `mode` | `"scan"` | One of: `scan` · `lock` · `monitor` |

<br>

| Mode | Role | Outputs used |
|------|------|-------------|
| `scan` | Generates cavity scan ramp + trigger square wave | OUT2 → ramp, OUT1 → trigger |
| `lock` | Applies PID feedback to laser current mod inputs | OUT1 → Slave1, OUT2 → Slave2 |
| `monitor` | Passive cavity signal monitor (no outputs) | — |

In [1]:
# ── Edit here ────────────────────────────────────────────────────────────────
BOARDS = {
    "Cav"  : {"ip": "192.168.0.99", "mode": "scan"},
    # "Lock1": {"ip": "192.168.0.102", "mode": "lock"},
    # "Mon"  : {"ip": "192.168.0.100", "mode": "monitor"},
}

SSH_USER       = "root"
SSH_PASS       = "root"
SSH_PORT       = 22

STCL_CMD_PORT  = 5000   # RP_Server command listener
STCL_LOOP_PORT = 5065   # reaction_loop port (opened during lock / scan)

# ── Auto-fix options ─────────────────────────────────────────────────────────
# Set to False to disable a specific auto-fix category
AUTO_INSTALL_PC_PACKAGES = True   # pip-install missing PC packages
AUTO_UPLOAD_STCL_FILES   = True   # SFTP-upload missing RP_side files to board
AUTO_KILL_STALE_PROCS    = True   # kill stale RunLock.py processes on board
AUTO_STOP_SCPI_SERVICE   = False  # stop redpitaya_scpi if it conflicts (disabled by default)

---
## 2) PC Auto-Install & Import Checker Module

This cell:
1. Checks for each required PC package and **installs it with `pip` if missing**.
2. Verifies the `Qt5Agg` matplotlib backend (installs `PyQt5` if absent).
3. Locates the repo root, adds it to `sys.path`, and imports `setup.py`.

> **Note:** If a package is freshly installed you may see a kernel-restart prompt.
> Re-run this cell once after restarting — no further installs will be triggered.

In [2]:
import sys, subprocess, importlib, pathlib

# ── Helpers ───────────────────────────────────────────────────────────────────
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"

def _pip_install(pkg_install_name, import_name=None):
    """Install *pkg_install_name* via pip and return True on success."""
    import_name = import_name or pkg_install_name
    print(f"{YELLOW}  ↳ Installing {pkg_install_name} …{RESET}", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg_install_name],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"{GREEN}    ✓ Installed {pkg_install_name}{RESET}")
        return True
    else:
        print(f"{RED}    ✗ Failed to install {pkg_install_name}:{RESET}")
        print(result.stderr.strip())
        return False

def ensure_package(import_name, pip_name=None):
    """Import *import_name*; auto-install *pip_name* if missing."""
    pip_name = pip_name or import_name
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "?")
        print(f"{GREEN}  ✓ {import_name:<18} already installed  (v{ver}){RESET}")
        return True
    except ImportError:
        if not AUTO_INSTALL_PC_PACKAGES:
            print(f"{RED}  ✗ {import_name:<18} NOT found  (auto-install disabled){RESET}")
            return False
        if _pip_install(pip_name, import_name):
            importlib.invalidate_caches()
            try:
                mod = importlib.import_module(import_name)
                ver = getattr(mod, "__version__", "?")
                print(f"{GREEN}  ✓ {import_name:<18} now available  (v{ver}){RESET}")
                return True
            except ImportError:
                print(f"{RED}  ✗ {import_name:<18} still not importable — restart kernel?{RESET}")
                return False
        return False

# ── Required PC packages ──────────────────────────────────────────────────────
print("─" * 68)
print("  PC PACKAGE AUTO-INSTALL")
print("─" * 68)

# (import_name, pip_install_name)
REQUIRED_PACKAGES = [
    ("paramiko",    "paramiko"),
    ("numpy",       "numpy"),
    ("scipy",       "scipy"),
    ("matplotlib",  "matplotlib"),
]

all_ok = all(ensure_package(imp, pip) for imp, pip in REQUIRED_PACKAGES)

# ── Qt5Agg backend (PyQt5) ────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")  # temporary non-GUI backend for the availability test
import matplotlib.pyplot as _plt

_qt5_available = False
try:
    import PyQt5  # noqa: F401
    _qt5_available = True
    print(f"{GREEN}  ✓ PyQt5              already installed{RESET}")
except ImportError:
    if AUTO_INSTALL_PC_PACKAGES:
        _qt5_available = _pip_install("PyQt5", "PyQt5")
    else:
        print(f"{YELLOW}  ⚠ PyQt5 not found — Qt5Agg backend unavailable (auto-install disabled){RESET}")

if not _qt5_available:
    print(f"{YELLOW}  ⚠ Live monitor windows may not work without Qt5Agg{RESET}")

print()

# ── Locate repo root and import setup.py ─────────────────────────────────────
print("─" * 68)
print("  IMPORTING CHECKER MODULE (setup.py)")
print("─" * 68)

_nb_dir = pathlib.Path().resolve()
for _candidate in [_nb_dir] + list(_nb_dir.parents)[:3]:
    if (_candidate / "setup.py").exists() and (_candidate / "lockclient.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import setup
importlib.reload(setup)
print(f"{GREEN}  ✓ setup.py loaded from: {pathlib.Path(setup.__file__).resolve()}{RESET}")

────────────────────────────────────────────────────────────────────
  PC PACKAGE AUTO-INSTALL
────────────────────────────────────────────────────────────────────
  ✓ paramiko           already installed  (v4.0.0)
  ✓ numpy              already installed  (v2.4.3)
  ✓ scipy              already installed  (v1.17.1)
  ✓ matplotlib         already installed  (v3.10.8)
  ✓ PyQt5              already installed

────────────────────────────────────────────────────────────────────
  IMPORTING CHECKER MODULE (setup.py)
────────────────────────────────────────────────────────────────────
  ✓ setup.py loaded from: C:\Users\Qulabs\Projects\RP-STCL\setup.py


---
## 3) PC Environment Checks

Verifies the Python version, all required packages, the `Qt5Agg` matplotlib
backend, and that the repository is correctly importable.

| Package | Used for |
|---------|----------|
| `paramiko` | SSH/SFTP — upload scripts and start server on each RP |
| `numpy` | Signal processing and array operations |
| `scipy` | Golden-ratio figure sizing; Savitzky-Golay filter math |
| `matplotlib` | Qt5Agg backend for live cavity and error monitor windows |

<br>

> All packages should now be present after Step 2. If any `✗` items remain here,
> check pip output above or install manually.

In [3]:
pc_ok = setup.check_pc()

────────────────────────────────────────────────────────────────────
  PC ENVIRONMENT CHECKS
────────────────────────────────────────────────────────────────────
  ✓ Python version  →  3.14.3 (>= 3.7 required for f-strings and other PC-side syntax)
  ✓ Package: paramiko      →  v4.0.0  —  SSH/SFTP: uploads scripts and starts server on each RP
  ✓ Package: numpy         →  v2.4.3  —  Signal processing and array operations
  ✓ Package: scipy         →  v1.17.1  —  Golden-ratio figure sizing; Savitzky-Golay filter math
  ✓ Package: matplotlib    →  v3.10.8  —  Qt5Agg backend for live cavity and error monitor windows
  ✓ matplotlib backend  →  Qt5Agg available — live monitor windows will work
  ✓ Repo on sys.path  →  C:\Users\Qulabs\Projects\RP-STCL
  ✓ RP_side importable  →  peak_finders module imported successfully
  ✓ RP_side/RP_Lock.py            →  Main locking loop (uploaded to board)
  ✓ RP_side/RunLock.py            →  Entry point executed on board via SSH
  ✓ RP_side/libserver.py 

---
## 4) Board Information

<p>Connects to each board via SSH and collects full hardware and software details
<em>before</em> any pass/fail judgement is applied.</p>

<blockquote style="border-left:4px solid #2980b9; padding:6px 12px;
  background:#eaf4fb; color:#1a5276; border-radius:4px;">
  <strong>What is collected:</strong>
  OS version · RP ecosystem version · hostname · uptime · CPU · RAM ·
  Python version &amp; sys.path · numpy version · rp SWIG module path &amp; import test ·
  STCL files on board · port 5000/5065 status ·
  stale RunLock processes · active system services
</blockquote>

<br>

> **SSH note:** The board runs Ubuntu 22.04 (RP OS 2.x). Standard paramiko
> key-exchange works without any `disabled_algorithms` workaround.

In [4]:
import setup

board_infos = {}
for name, cfg in BOARDS.items():
    info = setup.collect_board_info(
        name, cfg["ip"], cfg["mode"],
        ssh_user=SSH_USER, ssh_pass=SSH_PASS, ssh_port=SSH_PORT,
        stcl_cmd_port=STCL_CMD_PORT, stcl_loop_port=STCL_LOOP_PORT,
    )
    board_infos[name] = info

────────────────────────────────────────────────────────────────────
  Board: Cav  (192.168.0.99)  mode=scan
────────────────────────────────────────────────────────────────────
  ✓ Ping 192.168.0.99  — reachable
  ✓ SSH login  root@192.168.0.99
  i OS                : Ubuntu 22.04.5 LTS
  i RP ecosystem      : 2.07-ffe70f24f
  i RP .version file  : 2.07
  i Hostname          : rp-f0c97c
  i Uptime            : up 1 hour, 22 minutes
  i CPU               : ARMv7 Processor rev 0 (v7l)
  i Memory            : 461 MB total, 91 MB used
  i Python            : Python 3.10.12  (/usr/bin/python3)
  i /opt/redpitaya/lib/python on board sys.path: NO
  i numpy             : 2.2.5
  i rp module path    : not found
  i rp module import  : ok
  i /opt/redpitaya/bin : present
  i /opt/redpitaya/fpga: present
  ✓ Board STCL file  : /root/RP_Lock.py
  ✓ Board STCL file  : /root/RunLock.py
  ✓ Board STCL file  : /root/libserver.py
  ✓ Board STCL file  : /root/peak_finders.py
  ✓ Port 5000 (cmd)  : free

---
## 5) Board Compatibility Checks + Auto-Fix

Evaluates each board against STCL requirements, then **automatically resolves fixable issues** before printing the final result.

| Symbol | Meaning |
|--------|--------|
| `✓` | **Pass**: requirement met |
| `⚠` | **Warning**: may work but needs attention |
| `✗` | **Fail**: must be resolved before running STCL |
| `🔧` | **Auto-fixed**: issue detected and resolved automatically |

<br>

**Auto-fix actions (controlled by flags in Cell 1):**

| Flag | Action |
|------|-------|
| `AUTO_UPLOAD_STCL_FILES` | SFTP-uploads any missing STCL files from `RP_side/` to `/root/` on the board |
| `AUTO_KILL_STALE_PROCS` | Kills stale `RunLock.py` processes via SSH |
| `AUTO_STOP_SCPI_SERVICE` | Stops `redpitaya_scpi` if it is active (off by default) |

<details>
<summary><strong>Manual fixes for non-auto items (click to expand)</strong></summary>

<br>

| Issue | Fix |
|-------|-----|
| RP OS version is 1.04 | Flash SD card with OS 2.x from `downloads.redpitaya.com` |
| `rp` module not found / import fails | Verify `/opt/redpitaya/lib/python/rp.py` exists; check FPGA overlay is loaded |
| numpy ≥ 2.0 on board | Use patched `peak_finders.py` from this repo (`np.mat` → `np.array`) |
| Port 5000 still in use after auto-kill | Reboot the board |

</details>

In [5]:
import paramiko, pathlib, sys

GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
CYAN   = "\033[94m"
RESET  = "\033[0m"
WRENCH = "🔧"

# ── Locate RP_side directory (same repo root logic as Cell 2) ─────────────────
_nb_dir = pathlib.Path().resolve()
_repo_root = None
for _candidate in [_nb_dir] + list(_nb_dir.parents)[:3]:
    if (_candidate / "setup.py").exists() and (_candidate / "lockclient.py").exists():
        _repo_root = _candidate
        break

STCL_FILES = ["RP_Lock.py", "RunLock.py", "libserver.py", "peak_finders.py"]

def _ssh_connect(ip):
    """Return an open paramiko SSHClient or None on failure."""
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        client.connect(ip, port=SSH_PORT, username=SSH_USER,
                       password=SSH_PASS, timeout=8)
        return client
    except Exception as exc:
        print(f"{RED}    SSH connect failed ({ip}): {exc}{RESET}")
        return None

def _ssh_run(client, cmd):
    """Run *cmd* over SSH and return (stdout, stderr, exit_code)."""
    _, stdout, stderr = client.exec_command(cmd, timeout=15)
    return stdout.read().decode().strip(), stderr.read().decode().strip(), stdout.channel.recv_exit_status()


def autofix_board(name, cfg, info):
    """
    Examine *info* (from collect_board_info) and attempt to fix any
    auto-fixable issues.  Returns True if the board now looks ready.
    """
    ip = cfg["ip"]
    fixed_anything = False

    print(f"\n{'─'*68}")
    print(f"  Auto-fix: {name}  ({ip})")
    print(f"{'─'*68}")

    # We will open SSH once and reuse it for all fixes
    ssh = None

    # ── 1. Upload missing STCL files ─────────────────────────────────────────
    if AUTO_UPLOAD_STCL_FILES and _repo_root is not None:
        rp_side = _repo_root / "RP_side"
        missing = []
        for fname in STCL_FILES:
            local = rp_side / fname
            # info dict keys are lowered board-check style; fall back to direct check
            board_has = info.get(f"stcl_{fname}", info.get("stcl_files", {}).get(fname, False))
            if not board_has and local.exists():
                missing.append((fname, local))

        if missing:
            if ssh is None:
                ssh = _ssh_connect(ip)
            if ssh:
                sftp = ssh.open_sftp()
                for fname, local_path in missing:
                    try:
                        sftp.put(str(local_path), f"/root/{fname}")
                        print(f"{GREEN}  {WRENCH} Uploaded {fname} → /root/{fname}{RESET}")
                        fixed_anything = True
                    except Exception as exc:
                        print(f"{RED}  ✗ Upload failed for {fname}: {exc}{RESET}")
                sftp.close()
        else:
            print(f"{GREEN}  ✓ All STCL files present on board — no upload needed{RESET}")
    elif AUTO_UPLOAD_STCL_FILES and _repo_root is None:
        print(f"{YELLOW}  ⚠ Could not locate repo root — skipping STCL file upload{RESET}")
    else:
        print(f"{CYAN}  i Auto-upload disabled (AUTO_UPLOAD_STCL_FILES=False){RESET}")

    # ── 2. Kill stale RunLock.py processes ────────────────────────────────────
    stale_pids = info.get("stale_runlock_pids", [])
    if stale_pids:
        if AUTO_KILL_STALE_PROCS:
            if ssh is None:
                ssh = _ssh_connect(ip)
            if ssh:
                out, err, rc = _ssh_run(ssh, "pkill -f RunLock.py; sleep 0.5; pgrep -f RunLock.py | wc -l")
                remaining = int(out.strip()) if out.strip().isdigit() else -1
                if remaining == 0:
                    print(f"{GREEN}  {WRENCH} Killed stale RunLock.py process(es) (PIDs: {stale_pids}){RESET}")
                    fixed_anything = True
                else:
                    print(f"{YELLOW}  ⚠ pkill sent but {remaining} RunLock process(es) may still be running{RESET}")
        else:
            print(f"{YELLOW}  ⚠ Stale RunLock PIDs {stale_pids} detected — kill manually: pkill -f RunLock.py{RESET}")
            print(f"{CYAN}    (Set AUTO_KILL_STALE_PROCS=True to auto-resolve){RESET}")
    else:
        print(f"{GREEN}  ✓ No stale RunLock.py processes{RESET}")

    # ── 3. Stop SCPI service if conflicting ───────────────────────────────────
    scpi_active = info.get("scpi_active", False)
    if scpi_active:
        if AUTO_STOP_SCPI_SERVICE:
            if ssh is None:
                ssh = _ssh_connect(ip)
            if ssh:
                _ssh_run(ssh, "systemctl stop redpitaya_scpi")
                out, _, _ = _ssh_run(ssh, "systemctl is-active redpitaya_scpi")
                if "inactive" in out:
                    print(f"{GREEN}  {WRENCH} Stopped redpitaya_scpi service{RESET}")
                    fixed_anything = True
                else:
                    print(f"{YELLOW}  ⚠ Could not stop redpitaya_scpi — stop manually: systemctl stop redpitaya_scpi{RESET}")
        else:
            print(f"{YELLOW}  ⚠ redpitaya_scpi is active — may conflict on port 5000{RESET}")
            print(f"{CYAN}    (Set AUTO_STOP_SCPI_SERVICE=True to auto-stop){RESET}")
    else:
        print(f"{GREEN}  ✓ redpitaya_scpi not running{RESET}")

    # ── Cleanup ───────────────────────────────────────────────────────────────
    if ssh:
        ssh.close()

    if fixed_anything:
        print(f"\n{YELLOW}  ↳ Issues were auto-fixed. Re-running compatibility check…{RESET}")

    return fixed_anything


# ── Run compatibility checks with auto-fix ────────────────────────────────────
board_ok_all = True
for name, info in board_infos.items():
    cfg = BOARDS[name]

    # Auto-fix pass
    autofix_board(name, cfg, info)

    # Re-collect info if fixes were applied (so check_board sees fresh state)
    fresh_info = setup.collect_board_info(
        name, cfg["ip"], cfg["mode"],
        ssh_user=SSH_USER, ssh_pass=SSH_PASS, ssh_port=SSH_PORT,
        stcl_cmd_port=STCL_CMD_PORT, stcl_loop_port=STCL_LOOP_PORT,
    )
    board_infos[name] = fresh_info   # update for summary

    ok = setup.check_board(fresh_info,
                            stcl_cmd_port=STCL_CMD_PORT,
                            stcl_loop_port=STCL_LOOP_PORT)
    board_ok_all = board_ok_all and ok


────────────────────────────────────────────────────────────────────
  Auto-fix: Cav  (192.168.0.99)
────────────────────────────────────────────────────────────────────
  ✓ All STCL files present on board — no upload needed
  ✓ No stale RunLock.py processes
  ✓ redpitaya_scpi not running
────────────────────────────────────────────────────────────────────
  Board: Cav  (192.168.0.99)  mode=scan
────────────────────────────────────────────────────────────────────
  ✓ Ping 192.168.0.99  — reachable
  ✓ SSH login  root@192.168.0.99
  i OS                : Ubuntu 22.04.5 LTS
  i RP ecosystem      : 2.07-ffe70f24f
  i RP .version file  : 2.07
  i Hostname          : rp-f0c97c
  i Uptime            : up 1 hour, 22 minutes
  i CPU               : ARMv7 Processor rev 0 (v7l)
  i Memory            : 461 MB total, 91 MB used
  i Python            : Python 3.10.12  (/usr/bin/python3)
  i /opt/redpitaya/lib/python on board sys.path: NO
  i numpy             : 2.2.5
  i rp module path    : not fo

---
## 6) Summary Report

Aggregates all results across PC and boards into a single final verdict.

- All `✗ FAIL` items must be resolved before attempting any locking workflow.
- `⚠ WARNING` items should be reviewed — they may cause subtle issues at runtime.
- A green `✓` at the bottom means the system is ready.

> After resolving any remaining issues, **restart the kernel and re-run all cells**
> to confirm a clean state.

In [6]:
system_ready = setup.print_summary()


════════════════════════════════════════════════════════════════════
  SUMMARY
════════════════════════════════════════════════════════════════════

  ✓ PC environment
    13 passed  |  0 warnings  |  0 failed

  ✗ Board: Cav  (192.168.0.99)
    12 passed  |  3 warnings  |  1 failed
    Failed:
        ✗ rp module  →  /opt/redpitaya/lib/python/rp.py not found. This file is part of the RedPitaya OS 2.x installation. Verify the board is running OS 2.x and /opt/redpitaya/lib/python/ exists.
    Warnings:
        ⚠ rp lib on board sys.path  →  /opt/redpitaya/lib/python NOT in python3 sys.path. RP_Lock.py uses PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH in the SSH launch command — this is handled automatically by communication.py start_host_server(). No manual action needed unless you launch RunLock.py by hand.
        ⚠ Board numpy version  →  v2.2.5 — numpy >= 2.0 removed np.mat (used in peak_finders.py). Confirm you are using the patched peak_finders.py from this repo (np.mat repl